# AutoGen Selector Teams

## Scenario: coordinate an EU checkout investigation

Northstar needs a defensible recommendation for a 38% EU conversion drop. A selector coordinates observability, deployment, customer-impact, analyst, and risk-review roles. **Safety boundary:** this team prepares a proposal; it cannot execute rollback, notify customers, or bypass application authorization.

**Outcome:** learn participant definitions, selector logic, shared context, termination, ownership, failure recovery, tracing, and when a team earns its coordination cost.

![Multi-agent topology](../../../assets/multi-agent-patterns.svg)

A selector chooses the next contributor based on an unresolved evidence gap. The application, not the selector model, constrains eligible participants, scopes, budgets, termination, identity, and action authority.

## 1. Team contract before framework code

Required deliverable: a likely cause, evidence IDs, affected segment, uncertainty, mitigation proposal, rollback risk, and approval/escalation status. Ownership: Observability owns metrics/logs; Deployment owns releases; Customer Impact owns tickets/segments; Analyst synthesizes attributed evidence; Risk Reviewer challenges unsupported claims. This avoids the common pattern where five agents all vaguely ‘investigate.’

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'curriculum').exists():
    ROOT=next(p for p in (ROOT,*ROOT.parents) if (p/'curriculum').exists())
sys.path.insert(0,str(ROOT/'curriculum'/'advanced'/'05-incident-response-capstone'))
from agentops_lab.autogen_selector_team import run_selector_team,run_failure_loop,ownership_rules,MAX_TEAM_MESSAGES,MAX_AGENT_TURNS

run=run_selector_team()
print('stop:',run.stopped_reason)
for m in run.messages: print(f'{m.speaker}: {m.content}')
assert run.stopped_reason=='recommendation_ready'

stop: recommendation_ready
observability: Metrics show eu-west checkout-to-payment redirect down 38%; logs show 3DS callback errors.
deployment: A eu-west checkout UI release changed VAT validation and 3DS redirect handling before the drop.
customer_impact: Enterprise VAT-registered EU customers are most affected; support tickets mention redirect loops.
incident_analyst: Likely cause is the eu-west VAT/3DS UI change; recommend rollback or feature flag disablement.
risk_reviewer: Phrase as likely cause, verify rollback safety, and monitor conversion recovery before broad notification.
incident_analyst: Likely cause is the eu-west VAT/3DS UI change; recommend rollback or feature flag disablement.


## 2. SelectorGroupChat mapping

In AutoGen AgentChat, `AssistantAgent` participants contribute to a `SelectorGroupChat`. Participant descriptions help the selector distinguish roles. A selector prompt should choose an eligible specialist for a named evidence gap, prohibit redundant handoff, and stop only on a structured final deliverable. Combine semantic termination with `MaxMessageTermination`; retain per-agent, tool, time, and cost limits in application code.

Optional SDK shape: `SelectorGroupChat(participants=..., model_client=..., selector_prompt=..., termination_condition=TextMentionTermination('FINAL') | MaxMessageTermination(12))`. Keep credentials optional; this notebook uses a deterministic simulation.

In [2]:
print('ownership:')
for item in ownership_rules(): print('-',item)

# Deliberate coordination failure: agents bounce responsibility instead of closing evidence gaps.
failure=run_failure_loop()
print('failure stop:',failure.stopped_reason,'messages:',len(failure.messages),'turns:',failure.turns_by_agent)
assert failure.stopped_reason in {'MAX_TEAM_MESSAGES','observability reached MAX_AGENT_TURNS','deployment reached MAX_AGENT_TURNS','incident_analyst reached MAX_AGENT_TURNS'}

ownership:
- observability: Owns metrics and logs. Must not diagnose deployments.
- deployment: Owns release history. Must not infer customer impact.
- customer_impact: Owns affected segments and support reports.
- incident_analyst: Synthesizes evidence and asks targeted follow-ups.
- risk_reviewer: Challenges unsupported recommendations and risk.
failure stop: MAX_TEAM_MESSAGES messages: 12 turns: {'observability': 4, 'deployment': 4, 'incident_analyst': 4}


## 3. Review, evaluation, and production readiness

A reviewer must challenge unsupported diagnosis, not merely polish prose. Require evidence IDs and uncertainty; send conflict/missing evidence to an explicit escalation/abstention path. Evaluate the team against a single-agent baseline on outcome/evidence correctness, unsafe proposal rate, tool use, token/cost, latency, duplicate work, selector accuracy, termination correctness, and conflict recovery.

**Exercises:** (1) filter selector candidates after an agent supplied its artifact; (2) require two evidence IDs before `FINAL`; (3) add a missing deployment event and escalate; (4) compare a simple incident where a single agent wins; (5) instrument a selector-decision trace with candidate set, reason, cost, and stop condition.

References: [AutoGen SelectorGroupChat](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/selector-group-chat.html), [AutoGen paper](https://arxiv.org/abs/2308.08155).